In [1]:
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt
from scipy.stats import median_abs_deviation
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# ── Change this to your captured CSV filename ─────────────────────────────
INPUT_FILE  = "swim_20260430_153759.csv"
OUTPUT_FILE = f"processed/{INPUT_FILE}.csv"
 
# ── Wheel physical constants ──────────────────────────────────────────────
WHEEL_DIAMETER_M = 0.100                          # measure your actual wheel
WHEEL_CIRCUM_M   = np.pi * WHEEL_DIAMETER_M       # = 0.3142 m
COUNTS_PER_REV   = 4096                           # AS5600 is 12-bit
METERS_PER_COUNT = WHEEL_CIRCUM_M / COUNTS_PER_REV  # = 0.0000767 m per count
 
# ── Filter settings ───────────────────────────────────────────────────────
TARGET_FS_HZ  = 50.0    # resample to this uniform rate before filtering
CUTOFF_HZ     = 2.0     # low-pass cutoff — keeps stroke cycles, kills jitter
FILTER_ORDER  = 4       # Butterworth order
MAD_THRESHOLD = 6.0     # spike clip: how many MADs from median to allow

In [ ]:
# %% Step 1 — Load raw data
# ─────────────────────────────────────────────────────────────────────────
 
df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} rows")
print(df.head())
 
# Drop rows where the magnet wasn't detected.
# An AS5600 without a magnet returns garbage counts — including them would
# create phantom motion. Better to drop and interpolate over the gap.
n_bad = (df["magnet_ok"] == 0).sum()
if n_bad > 0:
    print(f"Dropping {n_bad} rows with magnet_ok=0")
df = df[df["magnet_ok"] == 1].copy().reset_index(drop=True)

Loaded 10879 rows
   timestamp_us  angle_counts  magnet_ok
0        100700          1383          1
1        104600          1383          1
2        108476          1383          1
3        112192          1383          1
4        115908          1383          1


In [ ]:
# %% Step 2 — Convert timestamp to seconds

df["time_s"] = (df["timestamp_us"] - df["timestamp_us"].iloc[0]) / 1e6
print(f"Session duration: {df['time_s'].iloc[-1]:.2f} s")
 

Session duration: 40.48 s


In [ ]:
# %% Step 3 — Unwrap the angle to remove rollover discontinuities

angle_rad      = df["angle_counts"].values * (2 * np.pi / COUNTS_PER_REV)
angle_unwrapped_rad = np.unwrap(angle_rad)
angle_unwrapped_counts = angle_unwrapped_rad * (COUNTS_PER_REV / (2 * np.pi))

In [6]:
# %% Step 4 — Convert counts to distance, fix sign
# ─────────────────────────────────────────────────────────────────────────
# Multiply unwrapped counts by meters-per-count to get cumulative distance.
# Subtract the starting value so distance begins at 0.
#
# Then negate: the wheel was mounted backwards, so all displacement was
# negative. Negating dist makes forward motion positive throughout.
 
dist_raw = angle_unwrapped_counts * METERS_PER_COUNT
dist_raw -= dist_raw[0]      # zero the start
if dist_raw[-1] < 0:
    dist_m = -dist_raw   # wheel mounted backwards
else:
    dist_m = dist_raw    # wheel mounted correctly

In [7]:
# %% Step 5 — Derive velocity using actual (non-uniform) dt
# ─────────────────────────────────────────────────────────────────────────
# The firmware had delay(50ms) but I2C reads add variable overhead, so
# actual dt per sample ranges from ~35ms to ~97ms.
#
# np.gradient(y, x) computes a central-difference derivative that uses the
# ACTUAL spacing between x values — it does not assume uniform sampling.
# This is critical: a fixed-dt assumption would inflate speed during short
# intervals and understate it during long ones.
 
t = df["time_s"].values
vel_raw = np.gradient(dist_m, t)    # m/s, non-uniform dt aware

In [8]:
# %% Step 6 — Remove spikes (before filtering)
# 
# MAD fails when the signal has many zeros (stationary periods) because
# the median and MAD both collapse to ~0, making the threshold ~0 and
# clipping all real motion.
#
# Fix: use a hard physical clamp instead.
# A 10cm wheel on a swimmer physically cannot exceed ~3 m/s.
# Anything beyond that is a bad sample — clip it unconditionally.

MAX_PHYSICAL_MS = 3.0
vel_clipped = np.clip(vel_raw, -MAX_PHYSICAL_MS, MAX_PHYSICAL_MS)

n_clipped = np.sum(np.abs(vel_raw) > MAX_PHYSICAL_MS)
print(f"Spike clip: {n_clipped} samples clipped ({n_clipped/len(vel_raw)*100:.1f}%)")
print(f"Vel after clip: min={vel_clipped.min():.3f}  max={vel_clipped.max():.3f}")

Spike clip: 0 samples clipped (0.0%)
Vel after clip: min=-0.991  max=1.196


In [9]:
# %% Step 7 — Interpolate to a uniform time grid
# ─────────────────────────────────────────────────────────────────────────
# The Butterworth filter (scipy.signal.butter + filtfilt) mathematically
# assumes evenly-spaced samples. Feeding it irregular data produces
# incorrect frequency response — the filter won't behave as designed.
#
# We interpolate to a clean uniform grid at TARGET_FS_HZ (50 Hz) using
# linear interpolation. This is safe here because the original data is
# already close to 50 Hz — we're filling small timing gaps, not upsampling
# coarse data.
 
t_uniform = np.arange(t[0], t[-1], 1.0 / TARGET_FS_HZ)
vel_uniform = np.interp(t_uniform, t, vel_clipped)

In [10]:
# %% Step 8 — Zero-phase low-pass filter
# Two cutoffs: one for velocity display, one for acceleration.
# Velocity wants higher cutoff to preserve stroke shape.
# Acceleration wants lower cutoff because differentiation amplifies
# any remaining noise by 2π × frequency — even small ripples become spikes.

VEL_CUTOFF_HZ   = (TARGET_FS_HZ / 2) * 0.10   # ~13.5 Hz — preserves stroke detail
ACCEL_CUTOFF_HZ = (TARGET_FS_HZ / 2) * 0.02   # ~2.7 Hz  — smooth enough to differentiate

b_vel,   a_vel   = butter(FILTER_ORDER, VEL_CUTOFF_HZ   / (TARGET_FS_HZ / 2), btype="low")
b_accel, a_accel = butter(FILTER_ORDER, ACCEL_CUTOFF_HZ / (TARGET_FS_HZ / 2), btype="low")

vel_filt       = filtfilt(b_vel,   a_vel,   vel_uniform)   # for plotting
vel_for_accel  = filtfilt(b_accel, a_accel, vel_uniform)   # for differentiation only

# %% Step 9 — Derive acceleration from the more aggressively filtered velocity
accel_filt = np.gradient(vel_for_accel, 1.0 / TARGET_FS_HZ)

print(f"Vel cutoff:   {VEL_CUTOFF_HZ:.2f} Hz")
print(f"Accel cutoff: {ACCEL_CUTOFF_HZ:.2f} Hz")
print(f"Vel range:    {vel_filt.min():.3f} to {vel_filt.max():.3f} m/s")
print(f"Accel range:  {accel_filt.min():.3f} to {accel_filt.max():.3f} m/s²")

Vel cutoff:   2.50 Hz
Accel cutoff: 0.50 Hz
Vel range:    -0.799 to 0.828 m/s
Accel range:  -0.600 to 0.610 m/s²


In [11]:
# %% Step 10 — Assemble and export
# ─────────────────────────────────────────────────────────────────────────
out = pd.DataFrame({
    "time_s":    t_uniform,
    "dist_m":    np.interp(t_uniform, t, dist_m),
    "vel_ms":    vel_filt,
    "accel_ms2": accel_filt,
})
out.to_csv(OUTPUT_FILE, index=False, float_format="%.4f")
print(f"\nExported {len(out)} rows → {OUTPUT_FILE}")
print(f"  Peak velocity:      {vel_filt.max():.3f} m/s")
print(f"  Peak acceleration:  {accel_filt.max():.3f} m/s²")
print(f"  Total distance:     {np.interp(t_uniform, t, dist_m).max():.3f} m")


Exported 2024 rows → processed/swim_20260430_153759.csv.csv
  Peak velocity:      0.828 m/s
  Peak acceleration:  0.610 m/s²
  Total distance:     6.930 m


In [12]:
# %% Step 11 — Plot (interactive, Plotly)
# ─────────────────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=("Distance (m)", "Velocity (m/s)", "Acceleration (m/s²)"),
    vertical_spacing=0.08,
)
 
# Distance
fig.add_trace(go.Scatter(
    x=out["time_s"], y=out["dist_m"],
    name="Distance", line=dict(color="#1d9e75", width=1.5)
), row=1, col=1)
 
# Velocity — raw clipped vs filtered
fig.add_trace(go.Scatter(
    x=t, y=vel_clipped,
    name="Vel raw (clipped)", line=dict(color="#b4b2a9", width=0.8),
    opacity=0.5
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=out["time_s"], y=out["vel_ms"],
    name="Vel filtered", line=dict(color="#185fa5", width=2)
), row=2, col=1)
 
# Acceleration — with fill for propulsive vs resistive
fig.add_trace(go.Scatter(
    x=out["time_s"], y=out["accel_ms2"],
    name="Acceleration", line=dict(color="#d85a30", width=1.5),
    fill="tozeroy",
    fillcolor="rgba(24,95,165,0.10)",
), row=3, col=1)
fig.add_hline(y=0, line=dict(color="#888780", width=0.8, dash="dot"), row=3, col=1)
 
fig.update_layout(
    title="AS5600 Swim Data — Cleaned Pipeline",
    height=700,
    template="plotly_white",
    legend=dict(orientation="h", y=-0.08),
    font=dict(size=11),
)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
 
fig.show()